# Análise de relevância: Vetorial × BM25 × LSA

Compara os três modelos que ranqueiam sobre o corpus completo do FAQ do Banco Central
(1.673 documentos):

1. **Consultas desafiadoras:** Top-5 de cada modelo lado a lado, em cinco consultas escolhidas
   pelo grupo (termo muito frequente, termo raro, sinônimo, polissemia e termo fora do
   vocabulário). A análise de cada uma se baseia **somente** no que foi retornado.
2. **Métricas:** MRR@10 e Recall@10 nas 373 consultas do dataset, incluindo a comparação do
   BM25 com `b = 0.75` × `b = 0`.

Busca Linear e Booleano ficam de fora: não ranqueiam (score fixo 1.0, na ordem do corpus) e
exigem todos os termos da consulta.

In [1]:
import os
import sys

# Executa a partir da raiz do projeto, esteja o notebook aberto em notebooks/ ou na raiz.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from collections import Counter

import pandas as pd

from src.buscador import criar_modelo
from src.dados import carregar_dados, montar_documentos
from src.preprocessamento import tokenizar_e_filtrar

pd.set_option("display.max_colwidth", 80)

In [2]:
df_corpus, _, _ = carregar_dados()
documentos = montar_documentos(df_corpus)

modelos = {nome: criar_modelo(nome) for nome in ("Vetorial", "BM25", "LSA")}
for modelo in modelos.values():
    modelo.indexar(documentos)

# Em quantos documentos cada termo aparece (frequência de documento).
frequencia_doc = Counter()
for texto in documentos.values():
    frequencia_doc.update(set(tokenizar_e_filtrar(texto)))

print(f"{len(documentos)} documentos indexados em {', '.join(modelos)}.")

1673 documentos indexados em Vetorial, BM25, LSA.


In [3]:
TOP_K = 5
TAMANHO_TRECHO = 70


def termos_da_consulta(consulta: str) -> pd.DataFrame:
    """Tokens da consulta e em quantos documentos cada um aparece."""
    tokens = tokenizar_e_filtrar(consulta)
    return pd.DataFrame(
        {"termo": tokens, "documentos_com_o_termo": [frequencia_doc[t] for t in tokens]}
    )


def top_k_lado_a_lado(consulta: str, k: int = TOP_K) -> pd.DataFrame:
    """Top-k de cada modelo em colunas: id, score e início do texto."""
    colunas = {}
    for nome, modelo in modelos.items():
        linhas = []
        for _id, score in modelo.buscar(consulta, k):
            trecho = " ".join(documentos[_id].split())[:TAMANHO_TRECHO]
            linhas.append(f"[{_id}] ({score:.3f}) {trecho}...")
        colunas[nome] = linhas + [""] * (k - len(linhas))
    return pd.DataFrame(colunas, index=pd.RangeIndex(1, k + 1, name="posição"))

## Consultas desafiadoras

### 1. Termo muito frequente

Consulta: **"o banco pode cobrar"**

In [4]:
consulta = "o banco pode cobrar"
termos_da_consulta(consulta)

,termo,documentos_com_o_termo
0,banco,455
1,pode,406
2,cobrar,26


In [5]:
top_k_lado_a_lado(consulta)

,Vetorial,BM25,LSA
posição,,,
1,"[a1173] (0.320) Sim, desde que você tenha pedido o cartão, o banco pode cobr...","[a1173] (8.712) Sim, desde que você tenha pedido o cartão, o banco pode cobr...",[a957] (0.588) Significa que o cheque somente pode ser pago mediante crédito...
2,"[a1018] (0.262) No caso de clientes, se o contrato incluir a realização de T...","[a1018] (7.645) No caso de clientes, se o contrato incluir a realização de T...",[a379] (0.550) Não. O Banco Central atua na esfera administrativa e não subs...
3,"[a1278] (0.255) Não. Para cliente pessoa física, existe uma lista de serviço...","[a1278] (7.632) Não. Para cliente pessoa física, existe uma lista de serviço...","[a1173] (0.528) Sim, desde que você tenha pedido o cartão, o banco pode cobr..."
4,"[a1431] (0.226) É uma operação de crédito, a exemplo do empréstimo, mas que ...","[a1431] (6.974) É uma operação de crédito, a exemplo do empréstimo, mas que ...",[a972] (0.501) É o cheque que somente pode ser pago em conta de titularidade...
5,[a854] (0.190) As instituições só podem cobrar as tarifas que estejam especi...,[a854] (6.124) As instituições só podem cobrar as tarifas que estejam especi...,"[a832] (0.498) O SCR pode ser consultado pelos próprios clientes, pelo Banco..."


**Análise.** "banco" e "pode" estão em 455 e 406 dos 1.673 documentos, ou seja, quase
não discriminam; o termo que decide é "cobrar" (26 documentos). Vetorial e BM25 devolvem
**o mesmo Top-5, na mesma ordem**, e 4 dos 5 falam de cobrança, tarifas ou serviços; o 1º
(a1173) responde literalmente "o banco pode cobrar tarifa pela emissão". O IDF baixo dos termos
frequentes fez o que devia: o ranking foi guiado por "cobrar". O LSA põe em 1º e 4º documentos
sobre **cheque cruzado** (a957, a972) e em 2º um sobre a atuação do Banco Central (a379), que
não falam de cobrança; a1173 só aparece em 3º. A leitura provável é que, no espaço latente,
os termos genéricos "banco"/"pode" pesaram mais do que "cobrar".

### 2. Termo raro

Consulta: **"atendimento prioritário para idosos e gestantes"**

In [6]:
consulta = "atendimento prioritário para idosos e gestantes"
termos_da_consulta(consulta)

,termo,documentos_com_o_termo
0,atendimento,92
1,prioritario,0
2,idosos,1
3,gestantes,1


In [7]:
top_k_lado_a_lado(consulta)

,Vetorial,BM25,LSA
posição,,,
1,"[a21] (0.330) Pessoas com deficiência ou com mobilidade reduzida, temporária...","[a21] (17.405) Pessoas com deficiência ou com mobilidade reduzida, temporári...",[a1045] (0.664) Sim. Você pode entrar em contato com o BC por meio dos segui...
2,[a911] (0.101) Não há regra que obrigue a oferta de atendimento presencial p...,[a911] (5.084) Não há regra que obrigue a oferta de atendimento presencial p...,[a581] (0.640) As reclamações podem ser registradas pelo site do BC seguindo...
3,[a1329] (0.077) O correspondente é canal de atendimento não obrigatório e pr...,[a1329] (4.635) O correspondente é canal de atendimento não obrigatório e pr...,[a1022] (0.586) Caso você possua uma conta de depósitos na instituição finan...
4,[a1022] (0.070) Caso você possua uma conta de depósitos na instituição finan...,[a1022] (4.432) Caso você possua uma conta de depósitos na instituição finan...,[a911] (0.545) Não há regra que obrigue a oferta de atendimento presencial p...
5,[a1017] (0.069) A responsabilidade é da instituição que contratou o correspo...,"[a721] (4.255) Caso queira, você pode entrar em contato com a instituição be...","[a1413] (0.543) Sim, as reclamações podem ser apresentadas pelos clientes e ..."


**Análise.** "idosos" e "gestantes" aparecem em **um único documento** do corpus (a21,
a lista de quem tem atendimento prioritário), e "prioritario" não aparece em nenhum.
Vetorial e BM25 colocam a21 em 1º com folga (BM25: 17,4 contra 5,1 do 2º), porque o IDF
máximo desses termos raros domina o score. O **LSA não traz a21 no Top-5**: devolve páginas
genéricas sobre canais de atendimento e reclamações (a1045, a581, a1413). A leitura provável
é que, com 100 componentes, termos presentes em um só documento quase não pesam nas direções
latentes, e sobra apenas o tema "atendimento". Isso é coerente com o MRR@10 mais baixo do
LSA (0,241).

### 3. Sinônimo

Consulta: **"empréstimo para comprar casa"**

In [8]:
consulta = "empréstimo para comprar casa"
termos_da_consulta(consulta)

,termo,documentos_com_o_termo
0,emprestimo,64
1,comprar,10
2,casa,9


In [9]:
top_k_lado_a_lado(consulta)

,Vetorial,BM25,LSA
posição,,,
1,[a1100] (0.271) Sim. Há duas formas de aderir a um grupo de consórcio em and...,[a1100] (9.065) Sim. Há duas formas de aderir a um grupo de consórcio em and...,[a610] (0.606) A principal diferença está na forma de pagamento da operação....
2,[a29] (0.251) Sim. Você poderá comprar qualquer bem que esteja no mesmo segm...,[a29] (8.831) Sim. Você poderá comprar qualquer bem que esteja no mesmo segm...,"[a1149] (0.606) É o valor máximo que pode ser descontado do salário, do bene..."
3,"[a1272] (0.216) O projeto de todas as cédulas da Segunda Família do Real, in...",[a1060] (7.509) O projeto da Segunda Família do Real foi desenvolvido pelo B...,[a450] (0.605) Existem dois tipos: empréstimo convencional e cartão de crédi...
4,[a1060] (0.215) O projeto da Segunda Família do Real foi desenvolvido pelo B...,"[a1272] (7.403) O projeto de todas as cédulas da Segunda Família do Real, in...","[a528] (0.591) Sim. Há convênios que liberam margem consignável ""extra"" para..."
5,"[a1069] (0.204) O projeto de todas as cédulas da Segunda Família do Real, in...","[a1069] (7.199) O projeto de todas as cédulas da Segunda Família do Real, in...","[a939] (0.567) Sempre que houver a liquidação de um empréstimo consignado, a..."


**Análise.** O corpus fala em "financiamento imobiliário" (22 documentos têm
"imobiliario"), e não em "empréstimo para comprar casa". **Nenhum modelo recupera um documento
de financiamento imobiliário no Top-5.** Vetorial e BM25 casam "comprar" com duas respostas
sobre **consórcio** (a1100, a29) e "casa" com três documentos sobre a **Casa da Moeda** e as
cédulas do Real (a1060, a1069, a1272): o termo certo no sentido errado. O LSA generaliza a
partir de "empréstimo" e traz cinco documentos sobre **empréstimo consignado** (a610, a1149,
a450, a528, a939): mais coerentes com o tema de crédito que os lexicais, mas ainda não é o
assunto pedido. Nenhum dos três modelos resolve a sinonímia nesse caso.

### 4. Polissemia

Consulta: **"como pagar a conta de luz"**

In [10]:
consulta = "como pagar a conta de luz"
termos_da_consulta(consulta)

,termo,documentos_com_o_termo
0,pagar,19
1,conta,286
2,luz,6


In [11]:
top_k_lado_a_lado(consulta)

,Vetorial,BM25,LSA
posição,,,
1,"[a970] (0.225) Não. O recebimento de contas de serviços públicos (água, luz,...",[a1289] (8.465) O empregado que possui conta-salário deve verificar junto ao...,[a482] (0.658) Não. Você precisará encerrar a sua conta atual e abrir uma co...
2,"[a576] (0.211) Sim. É possível à instituição financeira debitar, do limite d...","[a576] (8.233) Sim. É possível à instituição financeira debitar, do limite d...",[a1042] (0.650) Não. A conta-salário não é movimentável por cheques....
3,[a1289] (0.203) O empregado que possui conta-salário deve verificar junto ao...,"[a970] (7.731) Não. O recebimento de contas de serviços públicos (água, luz,...",[a1162] (0.644) Não. As normas de conta-salário não se aplicam a instituiçõe...
4,[a839] (0.174) Sim. O banco escolhido pelo empregador deve assegurar a dispo...,[a839] (6.455) Sim. O banco escolhido pelo empregador deve assegurar a dispo...,"[a787] (0.605) Sim, desde que você tenha pedido ou autorizado esse débito pr..."
5,[a1008] (0.163) Sim. Você pode pagar parte ou o total da dívida antecipadame...,"[a1199] (6.198) Sim, dependendo do que estiver pactuado em contrato. Há cont...",[a170] (0.598) Sim. O empregado pode manter apenas a conta-salário e não pre...


**Análise.** Aqui "conta" é **fatura**, mas no corpus quase sempre significa **conta
bancária** (286 documentos). O documento que responde é a970 ("o recebimento de contas de
serviços públicos (água, luz, telefone etc.)..."). O **Vetorial acerta em 1º**; o **BM25 põe
a970 em 3º**, atrás de dois documentos sobre conta-salário e cheque especial (a1289, a576),
que falam de "conta" no sentido bancário. O **LSA não traz a970**:
todo o Top-5 é sobre conta bancária e conta-salário (a482, a1042, a1162, a787, a170). Ficou
com o sentido dominante de "conta" no corpus, e "luz" (presente em apenas 6 documentos) não
influenciou o resultado.

### 5. Extra: termo fora do vocabulário

Consulta: **"bitcoin é regulado pelo banco central?"**

In [12]:
consulta = "bitcoin é regulado pelo banco central?"
termos_da_consulta(consulta)

,termo,documentos_com_o_termo
0,bitcoin,0
1,regulado,2
2,banco,455
3,central,312


In [13]:
top_k_lado_a_lado(consulta)

,Vetorial,BM25,LSA
posição,,,
1,[a1087] (0.204) O MPO surgiu a partir do Programa Nacional de Microcrédito P...,[a1087] (7.783) O MPO surgiu a partir do Programa Nacional de Microcrédito P...,[a379] (0.898) Não. O Banco Central atua na esfera administrativa e não subs...
2,[a1296] (0.183) As antecipações de recursos a exportadores brasileiros podem...,[a1296] (6.483) As antecipações de recursos a exportadores brasileiros podem...,[a132] (0.612) A duração do RAET é fixada pelo Banco Central no ato de sua d...
3,[a680] (0.124) Sim. O Banco Central e os bancos não cobram qualquer tarifa p...,[a1394] (4.424) O registro de capital estrangeiro no Banco Central tem como ...,[a1394] (0.611) O registro de capital estrangeiro no Banco Central tem como ...
4,[a893] (0.114) Operações cambiais ou transferências internacionais em reais ...,"[a614] (4.171) Essa cédula não pode ser substituída, depositada nem utilizad...",[a435] (0.600) Não. O Banco Central não regulamenta o tempo de espera em fil...
5,[a1394] (0.114) O registro de capital estrangeiro no Banco Central tem como ...,[a1295] (4.130) A forma mais fácil de acessar o SCR é por meio do Registrato...,"[a120] (0.572) No site do Banco Central, existe uma página dedicada a esse a..."


**Análise.** "bitcoin" não existe no corpus; o assunto aparece como "moedas virtuais" ou
"criptoativos" (9 documentos: a11, a447, a541, a638, a735, a809, a932, a1171, a1177). Como o
termo principal é descartado, sobram "banco", "central" (muito frequentes) e "regulado" (2
documentos), e **nenhum dos três modelos traz um documento sobre moedas virtuais**. Vetorial e
BM25 põem em 1º um texto sobre microcrédito (a1087), puxado por "regulado"; o LSA devolve
documentos genéricos sobre a atuação do Banco Central (a379, a435). É o limite dos três
modelos: nenhum deles consegue representar uma palavra que não viu na indexação.

## Métricas nas 373 consultas

Gerado por `python -m benchmark.avaliacao_relevancia`. Cada consulta tem exatamente um
documento relevante nos qrels, então o Recall@10 é a fração de consultas cujo documento
relevante aparece no Top-10.

In [14]:
relevancia = pd.read_csv(os.path.join("benchmark", "resultados", "relevancia.csv"))
relevancia.sort_values("mrr_10", ascending=False)

,modelo,mrr_10,recall_10,n_consultas
1,BM25,0.4041,0.6381,373
0,Vetorial,0.3768,0.6059,373
3,BM25 (b=0),0.3511,0.5576,373
2,LSA,0.2407,0.5201,373


**BM25 com `b = 0.75` × `b = 0`.** Com a normalização por comprimento (`b = 0.75`),
o BM25 é o melhor modelo nas duas métricas. Sem ela (`b = 0`), o MRR@10 cai de 0,404 para
0,351 e o Recall@10 de 63,8% para 55,8%, ficando abaixo do Vetorial. Nesse corpus, normalizar
pelo comprimento do documento ajuda.

## Conclusões

- **Vetorial e BM25 se comportam de forma parecida** nas consultas com termos frequentes e
  raros, e ambos acertam quando a consulta usa as mesmas palavras da resposta. O BM25 tem as
  melhores métricas no conjunto das 373 consultas.
- O **LSA perde termos raros e específicos** (consultas 2 e 4) e é o pior nas métricas.
  Na consulta 3 ele foi o único a trazer documentos do mesmo tema (crédito), mas ainda não
  o assunto certo.
- **Nenhum dos três modelos resolve sinônimos ausentes do corpus** (consultas 3 e 5): os
  lexicais casam a palavra no sentido errado ("Casa da Moeda") e o LSA não tem como
  representar termos fora do vocabulário.